# Video Game Dataset Analysis

This notebook has been adapted from the original Google Colab version so it can run from this GitHub repository.
Run the cells from top to bottom after installing the dependencies in `requirements.txt`.


In [ ]:
from pathlib import Path

DATA_PATH = Path("data/VideoGames.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/VideoGames.csv")

print(f"Using dataset: {DATA_PATH.resolve()}")


In [ ]:
#Import necessary libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, RocCurveDisplay)
#Set plot style
sns.set(style='whitegrid')

#Load the dataset
data = pd.read_csv(DATA_PATH)
data

#Display first few rows
data.head()
data.describe()
data.info()

In [ ]:
#Check for missing values
print(data.shape)
print('Missing Values:')
print(data.isnull().sum())

#Check data types
print('\nData Types:')
print(data.dtypes)

#Convert Critic_Score and User_Score to numeric, handling non-numeric values
#data['Critic_Score'] = pd.to_numeric(data['Critic_Score'], errors='coerce')
#data['User_Score'] = pd.to_numeric(data['User_Score'], errors='coerce')

#Impute missing scores with median
#imputer = SimpleImputer(strategy='mean')
#data[['Critic_Score', 'User_Score']] = imputer.fit_transform(data[['Critic_Score', 'User_Score']])
#data = data[data['Global_Sales'] <= 10].copy()
#Drop rows with missing sales data
data = data.dropna(subset=['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales','Critic_Score','User_Score'])

#Remove duplicates
data = data.drop_duplicates()

#Ensure sales are non-negative
data = data[(data['NA_Sales'] >= 0) & (data['EU_Sales'] >= 0) & (data['JP_Sales'] >= 0) &
            (data['Other_Sales'] >= 0) & (data['Global_Sales'] > 0)]

#Verify cleaning
print('\nAfter Cleaning - Missing Values:')
print(data.isnull().sum())
print('\nShape after cleaning:', data.shape)

In [ ]:
# Features and target
X = pd.get_dummies(data[['Critic_Score', 'User_Score', 'Genre']], columns=['Genre'], drop_first=True)
y = np.log1p(data['Global_Sales'])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=4349)

model_genre = LinearRegression().fit(X_train, y_train)

# --- metrics ---
r2_train = model_genre.score(X_train, y_train)
r2_test  = model_genre.score(X_test,  y_test)
y_pred = model_genre.predict(X_test)
n_train, p = X_train.shape
adj_r2_train = 1 - (1 - r2_train) * (n_train - 1) / (n_train - p - 1)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Train R²:        {r2_train:.4f}")
print(f"Train adj R²:    {adj_r2_train:.4f}")
print(f"Test  R²:        {r2_test:.4f}")
print(f"Test  RMSE:      {rmse:.4f}  (millions)")


# Plot predicted vs actual sales
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], linestyle='--')
plt.xlabel('Actual Global Sales (millions)')
plt.ylabel('Predicted Global Sales')
plt.title('Genre model – Predicted vs Actual on test set')
plt.grid(True)
plt.show()

In [ ]:
# Features and target
X = pd.get_dummies(data[['Year_of_Release']], drop_first=True)
y = np.log1p(data['User_Score'])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=502)

model_genre = LinearRegression().fit(X_train, y_train)

# --- metrics ---
r2_train = model_genre.score(X_train, y_train)
r2_test  = model_genre.score(X_test,  y_test)
y_pred = model_genre.predict(X_test)
n_train, p = X_train.shape
adj_r2_train = 1 - (1 - r2_train) * (n_train - 1) / (n_train - p - 1)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Train R²:        {r2_train:.4f}")
print(f"Train adj R²:    {adj_r2_train:.4f}")
print(f"Test  R²:        {r2_test:.4f}")
print(f"Test  RMSE:      {rmse:.4f}  (millions)")


# Plot predicted vs actual sales
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], linestyle='--')
plt.xlabel('Actual Global Sales (millions)')
plt.ylabel('Predicted Global Sales')
plt.title('Genre model – Predicted vs Actual on test set')
plt.grid(True)
plt.show()

In [ ]:
# Features and target
X = pd.get_dummies(data[['User_Score','Global_Sales','Platform', 'Genre','Year_of_Release']], drop_first=True)
y = (data['Critic_Score'])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=123)

model_genre = LinearRegression().fit(X_train, y_train)

# --- metrics ---
r2_train = model_genre.score(X_train, y_train)
r2_test  = model_genre.score(X_test,  y_test)
y_pred = model_genre.predict(X_test)
n_train, p = X_train.shape
adj_r2_train = 1 - (1 - r2_train) * (n_train - 1) / (n_train - p - 1)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Train R²:        {r2_train:.4f}")
print(f"Train adj R²:    {adj_r2_train:.4f}")
print(f"Test  R²:        {r2_test:.4f}")
print(f"Test  RMSE:      {rmse:.4f}  (Rating)")

intercept  = model_genre.intercept_
coeffs     = pd.Series(model_genre.coef_, index=X.columns)

# Nice, readable list
print("\nLinear‐model coefficients:")
print(f"Intercept = {intercept:.4f}")
for name, coef in coeffs.sort_values(key=abs, ascending=False).items():
    print(f"{name:<30} {coef:+.4f}")
# Plot predicted vs actual sales
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], linestyle='--')
plt.xlabel('Actual critic score')
plt.ylabel('Predicted critic score')
plt.title('Genre model – Predicted vs Actual on test set')
plt.grid(True)
plt.show()

In [ ]:
gp_stats = (data.groupby(['Genre'])
              .agg(avg_NA      = ('NA_Sales',     'mean'),
                   avg_EU      = ('EU_Sales',     'mean'),
                   avg_JP      = ('JP_Sales',     'mean'),
                   avg_Other   = ('Other_Sales',  'mean'),
                   avg_critic  = ('Critic_Score', 'mean'),
                   avg_user    = ('User_Score',   'mean'),
                   n_games     = ('Genre',        'size'))   # extra info
              .reset_index())

# 2. Feature matrix (numeric only for K-means) ------------------------------

X = gp_stats[[ 'avg_NA', 'avg_EU', 'avg_JP', 'avg_Other',
              'avg_critic', 'avg_user']]

# 3. Scale so each metric contributes equally -------------------------------

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Run K-means ------------------------------------------------------------

k       = 4                   # change 2…6 if you like
kmeans  = KMeans(n_clusters=k, random_state=0, n_init='auto')
labels  = kmeans.fit_predict(X_scaled)
gp_stats['cluster'] = labels

print("Silhouette score:", round(silhouette_score(X_scaled, labels), 3))

# 5. Detailed cluster view --------------------------------------------------

print("\nGenre × Platform clusters:")
for c in sorted(gp_stats['cluster'].unique()):
    print(f"\n── Cluster {c} ──")
    display(gp_stats.loc[gp_stats['cluster'] == c]
                      .sort_values('avg_user', ascending=False))

# 6. Quick numeric summary per cluster --------------------------------------

summary = (gp_stats.groupby('cluster')
                        .agg(pairs        = ('cluster',    'size'),
                             mean_NA      = ('avg_NA',     'mean'),
                             mean_EU      = ('avg_EU',     'mean'),
                             mean_JP      = ('avg_JP',     'mean'),
                             mean_Other   = ('avg_Other',  'mean'),
                             mean_cscore  = ('avg_critic', 'mean'),
                             mean_uscore  = ('avg_user',   'mean'))
                        .round(2))
print("\nCluster-level means:\n", summary)

In [ ]:
gp_stats = (data.groupby(['Genre', 'Platform'])
              .agg(
                   avg_NA      = ('NA_Sales',     'mean'),
                   avg_EU      = ('EU_Sales',     'mean'),
                   avg_JP      = ('JP_Sales',     'mean'),
                   avg_Other   = ('Other_Sales',  'mean'),
                   avg_critic  = ('Critic_Score', 'mean'),
                   avg_user    = ('User_Score',   'mean'),
                   n_games     = ('Genre',        'size'))   # games in pair
              .reset_index())

# 2. Feature matrix ---------------------------------------------------------
X = gp_stats[[ 'avg_NA', 'avg_EU', 'avg_JP', 'avg_Other',
              'avg_critic', 'avg_user']]

# 3. Standardise numeric features ------------------------------------------
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. K-means ---------------------------------------------------------------
k       = 4
kmeans  = KMeans(n_clusters=k, random_state=0, n_init='auto')
labels  = kmeans.fit_predict(X_scaled)
gp_stats['cluster'] = labels

print("Silhouette score:", round(silhouette_score(X_scaled, labels), 3))

# 5. For each cluster → show one row per Genre with its *dominant* platform
print("\nDominant platform per Genre (within each cluster):")
for c in sorted(gp_stats['cluster'].unique()):
    print(f"\n── Cluster {c} ──")
    clust = gp_stats.loc[gp_stats['cluster'] == c]

    # pick the platform with the most games for each genre
    dom = (clust.sort_values('n_games', ascending=False)
                 .groupby('Genre', as_index=False)
                 .first())   # keep the first row per genre (dom platform)

    display(dom[['Genre', 'Platform', 'n_games','avg_NA', 'avg_EU', 'avg_JP', 'avg_Other', 'avg_critic', 'avg_user']]
            .sort_values('avg_user', ascending=False))

# 6. Quick numeric summary per cluster --------------------------------------
summary = (gp_stats.groupby('cluster')
                        .agg(pairs        = ('cluster',    'size'),
                             mean_NA      = ('avg_NA',     'mean'),
                             mean_EU      = ('avg_EU',     'mean'),
                             mean_JP      = ('avg_JP',     'mean'),
                             mean_Other   = ('avg_Other',  'mean'),
                             mean_cscore  = ('avg_critic', 'mean'),
                             mean_uscore  = ('avg_user',   'mean'))
                        .round(2))
print("\nCluster-level means:\n", summary)

In [ ]:
MILESTONE = 0.6          # hit threshold (millions of copies)

# 1. Load & minimal clean -------------------------------------------------
df = pd.read_csv(DATA_PATH)
df['User_Score'] = pd.to_numeric(df['User_Score'], errors='coerce')
df = df[['Genre','Platform','Critic_Score','User_Score','Global_Sales']].dropna()

# 2. Binary label ---------------------------------------------------------
df['hit'] = (df['Global_Sales'] >= MILESTONE).astype(int)

# 3. Historical genre-average scores --------------------------------------
g_means = (df.groupby('Genre')
             .agg(genre_avg_critic=('Critic_Score', 'mean'),
                  genre_avg_user  =('User_Score',   'mean')))
df = df.join(g_means, on='Genre')

# 4. Feature matrix (one-hot cats) ----------------------------------------
X = pd.get_dummies(
        df[['Platform','Genre','genre_avg_critic','genre_avg_user']],
        columns=['Platform','Genre'],
        drop_first=True
    )
y = df['hit']

# Identify the numeric columns for scaling
num_cols = ['genre_avg_critic', 'genre_avg_user']

# 5. Train–test split -----------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y)

# 6. Standardise numeric columns (fit on train only) ----------------------
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

# 7. Fit logistic regression ---------------------------------------------
logreg = LogisticRegression(max_iter=500, class_weight='balanced')
logreg.fit(X_train, y_train)

# 8. Evaluation -----------------------------------------------------------
y_proba = logreg.predict_proba(X_test)[:, 1]
threshold = 0.5         # try 0.30, 0.20, etc. in a loop or ROC curve
y_pred = (y_proba >= threshold).astype(int)
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred-NoHit','Pred-Hit'],
            yticklabels=['True-NoHit','True-Hit'])
plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.show()

# Classification report
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))

# ROC–AUC
roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.3f}")
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title('ROC curve'); plt.show()

In [ ]:
df = pd.read_csv(DATA_PATH)
df['User_Score'] = pd.to_numeric(df['User_Score'], errors='coerce')

cols = ['Name','Genre','Platform','Critic_Score','User_Score',
        'Year_of_Release','Global_Sales']
df   = df[cols].dropna().reset_index(drop=True)

# 2. Feature matrix --------------------------------------------------------
X = pd.get_dummies(
        df[['Genre','Platform','Critic_Score','User_Score']],
        columns=['Genre','Platform'],
        drop_first=True)

y = np.log1p(df['Global_Sales'])          # work on log-sales for stability

# Identify numeric columns to scale
num_cols = ['Critic_Score','User_Score']
scaler   = StandardScaler()

# 3. Train–test split + scale ---------------------------------------------
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, df.index, test_size=0.30, random_state=42)

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

# 4. k-NN regressor --------------------------------------------------------
knn = KNeighborsRegressor(n_neighbors=60, weights='distance')
knn.fit(X_train, y_train)

# 5. Evaluation ------------------------------------------------------------
y_pred  = knn.predict(X_test)
mae_log = mean_absolute_error(y_test, y_pred)
r2      = r2_score(y_test, y_pred)

print(f"Test MAE (log units): {mae_log:.3f}")
print(f"Test R²:              {r2:.3f}")
print(f"≈ multiplicative MAE: {np.expm1(mae_log):.2%}")

# 6. Demo: predict a hypothetical new game ---------------------------------
new_game = pd.DataFrame({
    'Genre'        : ['Action'],
    'Platform'     : ['PS4'],
    'Critic_Score' : [85],
    'User_Score'   : [8.2]
})

newX = pd.get_dummies(new_game, columns=['Genre','Platform'], drop_first=True)
# align columns with training set (missing dummies → 0)
newX = newX.reindex(columns=X.columns, fill_value=0)

# scale numeric cols
newX[num_cols] = scaler.transform(newX[num_cols])

pred_log = knn.predict(newX)[0]
pred_lin = np.expm1(pred_log)
print(f"\nPredicted global sales for the new game ≈ {pred_lin:.2f} million units")

# 7. Show the 5 nearest neighbours ----------------------------------------
dists, neigh_idx = knn.kneighbors(newX, n_neighbors=5, return_distance=True)
print("\nNearest historical titles:")
for rank, (row_i, dist) in enumerate(zip(idx_train[neigh_idx[0]], dists[0]), 1):
    row = df.loc[row_i]
    print(f"{rank}. {row['Name']:<35}  "
          f"{row['Global_Sales']:.2f} M  "
          f"(Critic {row['Critic_Score']}, User {row['User_Score']})  "
          f"dist={dist:.2f}")

In [ ]:
# --------- K-means on (Genre × Decade) profiles ---------------------------
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 0.  Add a decade column ---------------------------------------------------
data = data.copy()
data['Decade'] = (data['Year_of_Release'] // 10) * 10    # e.g. 1993 → 1990

# 1.  Aggregate one row per (Genre, Decade) -------------------------------
gp_stats = (data.groupby(['Genre', 'Decade'])
              .agg(avg_NA     = ('NA_Sales',     'mean'),
                   avg_EU     = ('EU_Sales',     'mean'),
                   avg_JP     = ('JP_Sales',     'mean'),
                   avg_Other  = ('Other_Sales',  'mean'),
                   avg_critic = ('Critic_Score', 'mean'),
                   avg_user   = ('User_Score',   'mean'),
                   n_games    = ('Genre',        'size'))
              .reset_index())

# 2.  Feature matrix ------------------------------------------------------
X = gp_stats[['avg_NA','avg_EU','avg_JP','avg_Other',
              'avg_critic','avg_user']]

# 3.  Standardise ---------------------------------------------------------
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4.  K-means -------------------------------------------------------------
k       = 2                   # change to 2…6 and re-run if you like
kmeans  = KMeans(n_clusters=k, random_state=0, n_init='auto')
labels  = kmeans.fit_predict(X_scaled)
gp_stats['cluster'] = labels

print("Silhouette score:", round(silhouette_score(X_scaled, labels), 3))

# 5.  Detailed view -------------------------------------------------------
print("\nGenre × Decade clusters:")
for c in sorted(gp_stats['cluster'].unique()):
    print(f"\n── Cluster {c} ──")
    display(gp_stats.loc[gp_stats['cluster'] == c]
                      .sort_values('avg_user', ascending=False)
                      .reset_index(drop=True))

# 6.  Numeric summary -----------------------------------------------------
summary = (gp_stats.groupby('cluster')
                        .agg(pairs       = ('cluster',    'size'),
                             mean_NA     = ('avg_NA',     'mean'),
                             mean_EU     = ('avg_EU',     'mean'),
                             mean_JP     = ('avg_JP',     'mean'),
                             mean_Other  = ('avg_Other',  'mean'),
                             mean_cscore = ('avg_critic', 'mean'),
                             mean_uscore = ('avg_user',   'mean'))
                        .round(2))
print("\nCluster-level means:\n", summary)


In [ ]:
# Final K-Means run with k=2
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
data=pd.read_csv(DATA_PATH)

#Data Cleaning and Preparation
#Convert Critic_Score and User_Score to numeric, handle non-numeric values
data = data.copy()
data['Critic_Score'] = pd.to_numeric(data['Critic_Score'], errors='coerce')
data['User_Score'] = pd.to_numeric(data['User_Score'], errors='coerce')

#Add Decade column (e.g., 1993 → 1990)
#1996–2005 → 2000
#2006–2015 → 2010
#2016–2025 → 2020
data['Decade'] = (data['Year_of_Release'] // 10) * 10

#Aggregate one row per (Genre, Decade)
gp_stats = (data.groupby(['Genre', 'Decade'])
              .agg(avg_NA     = ('NA_Sales',     'mean'),
                   avg_EU     = ('EU_Sales',     'mean'),
                   avg_JP     = ('JP_Sales',     'mean'),
                   avg_Other  = ('Other_Sales',  'mean'),
                   avg_critic = ('Critic_Score', 'mean'),
                   avg_user   = ('User_Score',   'mean'),
                   n_games    = ('Genre',        'count'))
              .reset_index())

#Feature matrix for clustering
X = gp_stats[['avg_NA', 'avg_EU', 'avg_JP', 'avg_Other', 'avg_critic', 'avg_user']]

#Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#K-means clustering
k = 2  # Adjusted to 3 clusters for better separation; can experiment with 2–6
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
labels = kmeans.fit_predict(X_scaled)
gp_stats['cluster'] = labels

#Silhouette score
print("Silhouette score:", round(silhouette_score(X_scaled, labels), 3))

#Detailed view of clusters
print("\nGenre × Decade clusters:")
for c in sorted(gp_stats['cluster'].unique()):
    print(f"\n── Cluster {c} ──")
    display(gp_stats.loc[gp_stats['cluster'] == c]
                     .sort_values('avg_user', ascending=False)
                     .reset_index(drop=True))

#Numeric summary of clusters
summary = (gp_stats.groupby('cluster')
                   .agg(pairs       = ('cluster',    'size'),
                        mean_NA     = ('avg_NA',     'mean'),
                        mean_EU     = ('avg_EU',     'mean'),
                        mean_JP     = ('avg_JP',     'mean'),
                        mean_Other  = ('avg_Other',  'mean'),
                        mean_cscore = ('avg_critic', 'mean'),
                        mean_uscore = ('avg_user',   'mean'))
                   .round(2))
print("\nCluster-level means:\n", summary)

In [ ]:
# Scatter plot for the selected K-Means features
import matplotlib.pyplot as plt
plt.scatter(X_scaled[:, 0], X_scaled[:, 4], c=labels, cmap='viridis')
plt.xlabel('Scaled avg_NA')
plt.ylabel('Scaled avg_critic')
plt.title('K-means Clusters')
plt.show()

In [ ]:
# Elbow method and final K-Means run with k=3
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

#Load data
data = pd.read_csv(DATA_PATH)

#Data Cleaning and Preparation
data = data.copy()
data['Critic_Score'] = pd.to_numeric(data['Critic_Score'], errors='coerce')
data['User_Score'] = pd.to_numeric(data['User_Score'], errors='coerce')

#Add Decade column
data['Decade'] = (data['Year_of_Release'] // 10) * 10

#Aggregate one row per (Genre, Decade)
gp_stats = (data.groupby(['Genre', 'Decade'])
              .agg(avg_NA     = ('NA_Sales',     'mean'),
                   avg_EU     = ('EU_Sales',     'mean'),
                   avg_JP     = ('JP_Sales',     'mean'),
                   avg_Other  = ('Other_Sales',  'mean'),
                   avg_critic = ('Critic_Score', 'mean'),
                   avg_user   = ('User_Score',   'mean'),
                   n_games    = ('Genre',        'count'))
              .reset_index())

#Feature matrix for clustering
X = gp_stats[['avg_NA', 'avg_EU', 'avg_JP', 'avg_Other', 'avg_critic', 'avg_user']]

#Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Elbow Method
wcss = []
k_range = range(1, 11)  # Test k from 1 to 10
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)  # Inertia is WCSS

#Plot the Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.grid(True)
plt.show()

#Select k based on elbow and run final clustering (e.g., k=3 as an example)
k = 3  # Replace with the k value from the elbow plot
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
labels = kmeans.fit_predict(X_scaled)
gp_stats['cluster'] = labels

#Silhouette score
print("Silhouette score:", round(silhouette_score(X_scaled, labels), 3))

#Detailed view of clusters
print("\nGenre × Decade clusters:")
for c in sorted(gp_stats['cluster'].unique()):
    print(f"\n── Cluster {c} ──")
    display(gp_stats.loc[gp_stats['cluster'] == c]
                     .sort_values('avg_user', ascending=False)
                     .reset_index(drop=True))

#Numeric summary of clusters
summary = (gp_stats.groupby('cluster')
                   .agg(pairs       = ('cluster',    'size'),
                        mean_NA     = ('avg_NA',     'mean'),
                        mean_EU     = ('avg_EU',     'mean'),
                        mean_JP     = ('avg_JP',     'mean'),
                        mean_Other  = ('avg_Other',  'mean'),
                        mean_cscore = ('avg_critic', 'mean'),
                        mean_uscore = ('avg_user',   'mean'))
                   .round(2))
print("\nCluster-level means:\n", summary)

In [ ]:
# Summary statistics
print('Descriptive Statistics:')
print(data[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales',
            'Critic_Score', 'User_Score']].describe())

# Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(data[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales',
                  'Critic_Score', 'User_Score']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Distribution of Global Sales
plt.figure(figsize=(8, 6))
sns.histplot(data['Global_Sales'], bins=50, kde=True)
plt.title('Distribution of Global Sales')
plt.xlabel('Global Sales (Millions)')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Verify 'Decade' column exists; create it if not
if 'Decade' not in data.columns:
    data['Decade'] = (data['Year_of_Release'] // 10) * 10

#Check for missing values and data types
data = data.dropna(subset=['Global_Sales', 'Year_of_Release'])
print("Data Types:\n", data.dtypes)
print("Decade Values:\n", data['Decade'].unique())

#Group by Decade and compute mean for Global_Sales only
decade_sales = data.groupby('Decade')['Global_Sales'].mean().reset_index()

#Create the line plot
plt.figure(figsize=(12, 6))
sns.lineplot(x='Decade', y='Global_Sales', data=decade_sales)
plt.title('Average Global Sales by Decade')
plt.xlabel('Decade')
plt.ylabel('Average Global Sales (Millions)')
plt.show()